In [1]:
import json
import glob
import re
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

In [2]:
VSTATIONS_FILE = "../data/SNOWPACK/vstations.json"
SMET_DIR = "../data/SNOWPACK/smet"
OUTPUT_FILE = "../data/processed/snowpack_all.nc"

In [3]:
with open(VSTATIONS_FILE, "r") as f:
    vstations = json.load(f)

print("Antall stasjoner i json:", len(vstations))

Antall stasjoner i json: 6701


In [4]:
valid_stations = {
    station_id: meta
    for station_id, meta in vstations.items()
    if meta.get("error", "") == ""
}

print("Antall stasjoner uten modellfeil:", len(valid_stations))

Antall stasjoner uten modellfeil: 6638


In [5]:
valid_stations = {
    station_id: meta
    for station_id, meta in vstations.items()
    if meta.get("error", "") == ""
}

print("Antall stasjoner uten modellfeil:", len(valid_stations))

Antall stasjoner uten modellfeil: 6638


In [6]:
all_smet_files = sorted(glob.glob(f"{SMET_DIR}/*.smet"))

print("Totalt antall .smet-filer:", len(all_smet_files))

for f in all_smet_files[:20]:
    print(Path(f).name)

Totalt antall .smet-filer: 22721
VIR1000A.smet
VIR1000A1.smet
VIR1000A2.smet
VIR1000A3.smet
VIR1000A4.smet
VIR1001A.smet
VIR1001A1.smet
VIR1001A2.smet
VIR1001A3.smet
VIR1001A4.smet
VIR1002A.smet
VIR1002A1.smet
VIR1002A2.smet
VIR1002A3.smet
VIR1002A4.smet
VIR1003A.smet
VIR1003A1.smet
VIR1003A2.smet
VIR1003A3.smet
VIR1003A4.smet


In [7]:
base_smet_files = []

for f in all_smet_files:
    name = Path(f).stem
    if re.match(r"^VIR\d+A$", name):
        base_smet_files.append(f)

print("Antall basisfiler:", len(base_smet_files))
print("Eksempler:")
for f in base_smet_files[:10]:
    print(Path(f).name)

Antall basisfiler: 4541
Eksempler:
VIR1000A.smet
VIR1001A.smet
VIR1002A.smet
VIR1003A.smet
VIR1004A.smet
VIR1005A.smet
VIR1006A.smet
VIR1007A.smet
VIR1008A.smet
VIR1009A.smet


In [8]:
base_lookup = {Path(f).stem: f for f in base_smet_files}

In [9]:
stations_with_files = {
    station_id: meta
    for station_id, meta in valid_stations.items()
    if station_id in base_lookup
}

print("Stasjoner med både metadata og smet-fil:", len(stations_with_files))

Stasjoner med både metadata og smet-fil: 4486


In [10]:
def smet_to_dataset(station_id, meta):

    smet_file = base_lookup[station_id]

    with open(smet_file, "r") as f:
        lines = f.readlines()

    # finn kolonner
    fields_line = next(line for line in lines if line.startswith("fields"))
    columns = fields_line.split("=")[1].strip().split()

    # finn start på data
    data_start = next(i for i, line in enumerate(lines) if line.strip() == "[DATA]") + 1

    data_lines = [line.strip() for line in lines[data_start:] if line.strip()]
    rows = [line.split() for line in data_lines]

    df = pd.DataFrame(rows, columns=columns)

    # tid
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # numeriske kolonner
    for col in df.columns:
        if col != "timestamp":
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # erstatt Snowpack nodata med np.nan, ikke pd.NA
    df = df.replace(-999, np.nan)

    ds = xr.Dataset(
        {col: ("time", df[col].to_numpy(dtype=float)) for col in df.columns if col != "timestamp"},
        coords={"time": df["timestamp"].values}
    )

    ds = ds.expand_dims("station")

    ds = ds.assign_coords(
        station=("station", [station_id]),
        lon=("station", [meta["lon"]]),
        lat=("station", [meta["lat"]]),
        elev=("station", [meta["elev"]]),
        x=("station", [meta["easting"]]),
        y=("station", [meta["northing"]]),
    )

    return ds

In [11]:
station_id, meta = list(stations_with_files.items())[0]

test_ds = smet_to_dataset(station_id, meta)

print(test_ds)

<xarray.Dataset> Size: 3MB
Dimensions:               (station: 1, time: 6575)
Coordinates:
  * station               (station) <U5 20B 'VIR1A'
    lon                   (station) float64 8B 7.428
    lat                   (station) float64 8B 61.84
    elev                  (station) float64 8B 1.41e+03
    x                     (station) float64 8B 1.02e+05
    y                     (station) float64 8B 6.88e+06
  * time                  (time) datetime64[us] 53kB 2024-09-01T01:00:00 ... ...
Data variables: (12/63)
    Qs                    (station, time) float64 53kB 1.818 2.454 ... 0.078
    Ql                    (station, time) float64 53kB 0.419 0.437 ... 6.29 0.08
    Qg                    (station, time) float64 53kB 0.0 0.0 0.0 ... 0.0 0.0
    TSG                   (station, time) float64 53kB 0.0 0.0 0.0 ... 0.0 0.0
    Qg0                   (station, time) float64 53kB nan nan nan ... 0.0 0.0
    Qr                    (station, time) float64 53kB 0.0 0.0 0.0 ... 0.0 0.0
    

In [12]:
datasets = []
failed = []

total = len(stations_with_files)

for i, (station_id, meta) in enumerate(stations_with_files.items(), start=1):
    if i % 500 == 0:
        print(f"{i}/{total} stasjoner lest")

    try:
        ds = smet_to_dataset(station_id, meta)
        datasets.append(ds)
    except Exception as e:
        failed.append((station_id, str(e)))

print("Antall datasett:", len(datasets))
print("Antall feilet:", len(failed))
print(failed[:10])

500/4486 stasjoner lest
1000/4486 stasjoner lest
1500/4486 stasjoner lest
2000/4486 stasjoner lest
2500/4486 stasjoner lest
3000/4486 stasjoner lest
3500/4486 stasjoner lest
4000/4486 stasjoner lest
Antall datasett: 4484
Antall feilet: 2
[('VIR964A', ''), ('VIR978A', '')]


In [13]:
snowpack_all = xr.concat(datasets, dim="station")
print(snowpack_all)

/var/folders/4n/m1gxpf611c158s8wx2jzfqgw0000gn/T/ipykernel_3673/1777882233.py:1: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  snowpack_all = xr.concat(datasets, dim="station")


<xarray.Dataset> Size: 15GB
Dimensions:               (station: 4484, time: 6575)
Coordinates:
  * station               (station) <U8 143kB 'VIR1A' 'VIR2A' ... 'VIR6701A'
    lon                   (station) float64 36kB 7.428 7.426 ... 9.199 9.197
    lat                   (station) float64 36kB 61.84 61.85 ... 61.55 61.56
    elev                  (station) float64 36kB 1.41e+03 1.687e+03 ... 946.0
    x                     (station) float64 36kB 1.02e+05 1.02e+05 ... 1.92e+05
    y                     (station) float64 36kB 6.88e+06 ... 6.839e+06
  * time                  (time) datetime64[us] 53kB 2024-09-01T01:00:00 ... ...
Data variables: (12/63)
    Qs                    (station, time) float64 236MB 1.818 2.454 ... 5.28
    Ql                    (station, time) float64 236MB 0.419 0.437 ... 1.127
    Qg                    (station, time) float64 236MB 0.0 0.0 0.0 ... 0.0 0.0
    TSG                   (station, time) float64 236MB 0.0 0.0 0.0 ... 0.0 0.0
    Qg0                 

In [14]:
snowpack_all.to_netcdf(OUTPUT_FILE)
print("Lagret til:", OUTPUT_FILE)

Lagret til: ../data/processed/snowpack_all.nc
